# Integrate Google Agentspace with Gen AI Toolbox

This notebook provides a working example of integrating [Google Agentspace](https://cloud.google.com/products/agentspace?e=48754805&hl=en) with Google Cloud databases via [Gen AI Toolbox](https://github.com/googleapis/genai-toolbox). While this notebook uses Spanner as the source database, you can leverage the same pattern to integrate with [any database](https://googleapis.github.io/genai-toolbox/resources/sources/#available-sources) that Gen AI Toolbox supports.

## Basic Setup

## Pre-requisites

You will need to be allow-listed for Agentspace access before you can run the final steps in this notebook, so please work with your account team to gain access to the preview feature. However, you can run all of the steps up to and including the creation of an Agent Builder Conversational App that integrates with Spanner via Toolbox without being allow-listed.

### Install dependencies

In [ ]:
%pip install \
    langgraph==0.3.21 \
    langchain-google-vertexai==2.0.18 \
    toolbox-langchain==0.1.0 \
    --quiet

### Authenticate to Google Cloud within Colab
If you're running this on google colab notebook, you will need to Authenticate as an IAM user.

In [ ]:
from google.colab import auth

auth.authenticate_user()

### Define Notebook Parameters

In [ ]:
# @markdown Update the parameters below to match your environment.
# @markdown > Note: You can leave bucket names empty to create new GCS buckets.

# Please fill in these values.
project_id = "your-project"  # @param {type:"string"}
region = "your-region"  # @param {type:"string"}
vpc = "your-vpc"  # @param {type:"string"}
spanner_instance_id = "products-instance"  # @param {type:"string"}
spanner_database_id = "products-database"  # @param {type:"string"}
spanner_table_id = "products"  # @param {type:"string"}
spanner_avro_export_location = "gs://pr-public-demo-data/spanner-retail-demo/products-instance-orders-database/"  # @param {type:"string"}

spanner_session = None  # Global Spanner session variable
project_number = ! gcloud projects describe {project_id} --format='value(projectNumber)'
project_number = project_number[0]


### Connect Your Google Cloud Project

In [ ]:
# Configure gcloud.
!gcloud config set project {project_id}

### Enable APIs

In [ ]:
!gcloud services enable spanner.googleapis.com

### Configure Logging

In [ ]:
import logging
import sys

# Configure the root logger to output messages with INFO level or above
logging.basicConfig(level=logging.INFO, stream=sys.stdout, format='%(asctime)s[%(levelname)5s][%(name)14s] - %(message)s',  datefmt='%H:%M:%S', force=True)

### Helper Functions

#### rest_api_helper()

In [ ]:
import requests
import google.auth
import json

# Get an access token based upon the current user
creds, _ = google.auth.default()
authed_session = google.auth.transport.requests.AuthorizedSession(creds)
access_token=creds.token

if project_id:
  authed_session.headers.update({"x-goog-user-project": project_id}) # Required to workaround a project quota bug

def rest_api_helper(
    session: requests.Session,
    url: str,
    http_verb: str,
    request_body: dict = None,
    params: dict = None
  ) -> dict:
  """Calls a REST API using a pre-authenticated requests Session."""

  headers = {"Content-Type": "application/json"}

  try:

    if http_verb == "GET":
      response = session.get(url, headers=headers, params=params)
    elif http_verb == "POST":
      response = session.post(url, json=request_body, headers=headers, params=params)
    elif http_verb == "PUT":
      response = session.put(url, json=request_body, headers=headers, params=params)
    elif http_verb == "PATCH":
      response = session.patch(url, json=request_body, headers=headers, params=params)
    elif http_verb == "DELETE":
      response = session.delete(url, headers=headers, params=params)
    else:
      raise ValueError(f"Unknown HTTP verb: {http_verb}")

    # Raise an exception for bad status codes (4xx or 5xx)
    response.raise_for_status()

    # Check if response has content before trying to parse JSON
    if response.content:
        return response.json()
    else:
        return {} # Return empty dict for empty responses (like 204 No Content)

  except requests.exceptions.RequestException as e:
      # Catch potential requests library errors (network, timeout, etc.)
      # Log detailed error information
      print(f"Request failed: {e}")
      if e.response is not None:
          print(f"Request URL: {e.request.url}")
          print(f"Request Headers: {e.request.headers}")
          print(f"Request Body: {e.request.body}")
          print(f"Response Status: {e.response.status_code}")
          print(f"Response Text: {e.response.text}")
          # Re-raise a more specific error or a custom one
          raise RuntimeError(f"API call failed with status {e.response.status_code}: {e.response.text}") from e
      else:
          raise RuntimeError(f"API call failed: {e}") from e
  except json.JSONDecodeError as e:
      print(f"Failed to decode JSON response: {e}")
      print(f"Response Text: {response.text}")
      raise RuntimeError(f"Invalid JSON received from API: {response.text}") from e



## Setup Spanner

### Define Spanner Helper Functions

In [ ]:
def get_spanner_sessions(project_id = project_id, instance_id = spanner_instance_id, database_id = spanner_database_id):
  url = f"https://spanner.googleapis.com/v1/projects/{project_id}/instances/{instance_id}/databases/{database_id}/sessions"
  response = rest_api_helper(authed_session, url, "GET")
  return response

In [ ]:
# https://cloud.google.com/spanner/docs/reference/rest/v1/projects.instances.databases.sessions/create
def create_spanner_session(project_id = project_id, instance_id = spanner_instance_id, database_id = spanner_database_id):

  # Create a new session
  url = f"https://spanner.googleapis.com/v1/projects/{project_id}/instances/{instance_id}/databases/{database_id}/sessions"
  params = {
      "database": f"projects/{project_id}/instances/{instance_id}/databases/{database_id}"
  }
  response = rest_api_helper(authed_session, url, "POST", {}, params)
  return response['name']

In [ ]:
# https://cloud.google.com/spanner/docs/reference/rest/v1/projects.instances.databases.sessions/delete
def close_spanner_session(session, project_id = project_id, instance_id = spanner_instance_id, database_id = spanner_database_id):
  url = f"https://spanner.googleapis.com/v1/{session}"
  response = rest_api_helper(authed_session, url, "DELETE", {}, {"name": f"{session}"})
  return response

In [ ]:
import pandas as pd

def run_spanner_query(sql, database_id = spanner_database_id, query_options=None, create_new_session=False):
  """
  Runs a Spanner query and returns the result.

  Args:
      sql: The SQL query to execute.
      database_id: The database to query.
      query_options: (Optional) A dictionary of advanced query options.
                    See https://cloud.google.com/spanner/docs/reference/rest/v1/projects.instances.databases.sessions/executeSql#queryoptions
                    for available options.
      create_new_session: Defines whether to run the query in a new session.

  Returns:
      A dictionary containing the query results.

  Ref:
      https://cloud.google.com/spanner/docs/reference/rest/v1/projects.instances.databases.sessions/executeSql
  """
  # Ensure a spanner_session exists
  global spanner_session
  if not spanner_session or create_new_session == True:
    spanner_session = create_spanner_session()

  # Initialize response vars
  commit_response = ""
  response = ""

  # Construct the request URL
  uri = f"https://spanner.googleapis.com/v1/{spanner_session}:executeSql"

  # Set transaction type (readOnly/readWrite) and transaction object with commit type (begin/singleUse)
  transaction_type = "readWrite" if any(x in sql.lower() for x in ["insert", "update", "delete"]) else "readOnly"
  transaction = {"begin": {"readWrite": {}}} if transaction_type == "readWrite" else {"singleUse": {"readOnly": {}}}

  # Construct the request
  request_body = {
      "sql": sql,
      "transaction": transaction
  }
  params = {
      "session": spanner_session
  }

  if query_options:
      request_body["queryOptions"] = query_options

  try:
    # Make the request
    response = rest_api_helper(authed_session, uri, "POST", request_body=request_body, params = params)

  except RuntimeError as e:
    if "Session not found" in str(e):
      print(f"Session not found. Creating a new session and retrying the query...")
      return run_spanner_query(sql, database_id, query_options, create_new_session=True)  # Retry with a new session
    else:
      raise  # Re-raise the exception if it's not a "Session not found" error

  # Commit transaction if read/write
  if transaction_type == "readWrite":
      uri = f"https://spanner.googleapis.com/v1/{spanner_session}:commit"
      params = {
          "session": spanner_session
      }
      commit_response = rest_api_helper(authed_session, uri, "POST", {"transactionId": response['metadata']['transaction']['id']}, params)
      print(f"commit_response: {commit_response}")

  # Return a DataFrame if type is SELECT, WITH, or GRAPH
  if transaction_type == 'readOnly':
    columns = [field.get('name', 'unnamed_column') for field in response['metadata']['rowType']['fields']]

    # Create DataFrame from rows
    if 'rows' in response:
      df = pd.DataFrame(response['rows'], columns=columns)
      return df
    else:
      return response

  else:
    # Return the query results
    return response

In [ ]:
import time

def run_spanner_ddl(ddl_array, project_id = project_id, instance_id = spanner_instance_id, database_id = spanner_database_id):
  # https://cloud.google.com/spanner/docs/reference/rest/v1/projects.instances.databases.tables/create#try-it

  uri = f"https://spanner.googleapis.com/v1/projects/{project_id}/instances/{instance_id}/databases/{database_id}/ddl"
  http_verb = "PATCH"
  request_body = {
      "statements": ddl_array
  }

  response = rest_api_helper(authed_session, uri, http_verb, request_body)

  operation_name = response['name']
  uri = f"https://spanner.googleapis.com/v1/{operation_name}"

  while True:
    response = rest_api_helper(authed_session, uri, "GET", {})
    if response.get("done", False):
      if response.get("error"):
        print(response.get("error"))
      else:
        print("Operation completed successfully.")
      break
    else:
      print("Operation not completed yet.")
      time.sleep(2)


### Create Spanner Instance

In [ ]:
# Create the Spanner instance
# This notebook creates a paid instance. 90-day Free trial available once per project lifecycle
# https://cloud.google.com/spanner/docs/reference/rest/v1/projects.instances/create
url = f"https://spanner.googleapis.com/v1/projects/{project_id}/instances"
http_verb = "POST"
request_body = {
    "instance": {
        "config": f"projects/{project_id}/instanceConfigs/regional-us-central1",
        "displayName": f"{spanner_instance_id}",
        "edition": "ENTERPRISE",
        "processingUnits": 100,

        # OPTIONAL: Define nodeCount instead of processingUnits or autoscalingConfig.
        #"nodeCount": 1,

        # OPTIONAL: Define autoscalingConfig instead of nodeCount or processingUnits.
        #"autoscalingConfig": {
        #  "autoscalingLimits": {
        #    "minProcessingUnits": 1000,
        #    "maxProcessingUnits": 2000
        #  },
        #  "autoscalingTargets": {
        #    "highPriorityCpuUtilizationPercent": 80,
        #    "storageUtilizationPercent": 80
        #  }
        #}
    },
    "instanceId": f"{spanner_instance_id}"
}

response = rest_api_helper(authed_session, url, http_verb, request_body)
response

In [ ]:
# Create the Spanner database
# https://cloud.google.com/spanner/docs/reference/rest/v1/projects.instances.databases/create
url = f"https://spanner.googleapis.com/v1/projects/{project_id}/instances/{spanner_instance_id}/databases"
http_verb = "POST"
request_body = {
    "createStatement": f"CREATE DATABASE `{spanner_database_id}`",
    "databaseDialect": "GOOGLE_STANDARD_SQL"
}

response = rest_api_helper(authed_session, url, http_verb, request_body)
response

### Add Required Permissions

In [ ]:
roles_array = [
    "roles/spanner.viewer",
    "roles/dataflow.worker",
    "roles/storage.admin",
    "roles/spanner.databaseReader",
    "roles/spanner.databaseAdmin",
]

for r in roles_array:
  ! gcloud projects add-iam-policy-binding {project_id} \
      --member="serviceAccount:{project_number}-compute@developer.gserviceaccount.com" \
      --role="{r}"


### Enable Vertex AI Integration

In [ ]:
# Create and Embeddings Model and LLM Model
# Ref: https://codelabs.developers.google.com/codelabs/spanner-getting-started-vector-search#3
#      https://cloud.google.com/spanner/docs/ml-tutorial-embeddings
ddl_array = []
ddl_array.append(f"""CREATE MODEL IF NOT EXISTS LLMModel INPUT(
prompt STRING(MAX),
) OUTPUT(
content STRING(MAX),
) REMOTE OPTIONS (
endpoint = '//aiplatform.googleapis.com/projects/{project_id}/locations/us-central1/publishers/google/models/gemini-2.0-flash-001',
default_batch_size = 1
)
""")

ddl_array.append(f"""CREATE MODEL IF NOT EXISTS EmbeddingsModel INPUT(
  content STRING(MAX),
  ) OUTPUT(
  embeddings STRUCT<statistics STRUCT<truncated BOOL, token_count FLOAT64>, values ARRAY<FLOAT64>>,
  ) REMOTE OPTIONS (
  endpoint = '//aiplatform.googleapis.com/projects/{project_id}/locations/us-central1/publishers/google/models/text-embedding-005'
  )
""")

result = run_spanner_ddl(ddl_array)
result

### Test the Models

> NOTE: It may take a minute or two for the integration to complete in the background before the tests below will work.

In [ ]:
sql = """
SELECT embeddings.values
  FROM ML.PREDICT(
    MODEL EmbeddingsModel,
    (SELECT 'This is a test string that will be converted to an embedding' as content))
"""

run_spanner_query(sql)

In [ ]:
sql = """SELECT *
FROM ML.PREDICT(
MODEL LLMModel,
(   SELECT
'What is Google Agentspace?' AS prompt),
STRUCT(256 AS maxOutputTokens))"""

run_spanner_query(sql)

## Load Spanner Data

### Kick Off Import Job

In [ ]:
# Kick off the Dataflow import job
result = ! gcloud dataflow jobs run import-spanner \
    --gcs-location='gs://dataflow-templates-{region}/latest/GCS_Avro_to_Cloud_Spanner' \
    --region={region} \
    --parameters='instanceId={spanner_instance_id},databaseId={spanner_database_id},inputDir={spanner_avro_export_location}' \
    --network={vpc}

# Get id of Dataflow job from result
job_id = ""
for item in result:
  if item.startswith('id'):
    job_id = item.split()[1]

# Show result
result

### Wait for Import Completion

In [ ]:
import time

# Define Helper Function
def wait_for_dataflow_job(id: str):
  # Check status of Dataflow job
  job_state = ! gcloud dataflow jobs describe {job_id} --region={region} --format='value(currentState)'

  # Wait until Dataflow job is complete, checking status every 10 seconds
  while job_state[0] in ['JOB_STATE_RUNNING', 'JOB_STATE_PENDING']:
    print(f"Dataflow job {job_id} is in state: {job_state[0]}")
    time.sleep(10)
    job_state = ! gcloud dataflow jobs describe {job_id} --region={region} --format='value(currentState)'

  # Show final Dataflow job state
  print(f"Dataflow job {job_id} final state: {job_state[0]}")
  return job_state[0]

# Wait for job to complete
wait_for_dataflow_job(job_id)

## Generate Embeddings

### Add Embedding Columns

Reference: https://cloud.google.com/spanner/docs/backfill-embeddings

In [ ]:
# Add embedding column
ddl_array = []

ddl_array.append("ALTER TABLE products ADD COLUMN embedding ARRAY<FLOAT64>")
ddl_array.append("ALTER TABLE products ADD COLUMN embedding_model_version STRING(256)")

run_spanner_ddl(ddl_array)

### Backfill Embeddings

In [ ]:
count_sql = "SELECT COUNT(*) AS to_embed FROM products WHERE products.embedding IS NULL"
count_result = run_spanner_query(count_sql)
rows_left = int(count_result.values[0][0])

while rows_left > 0:
  sql = """UPDATE products
  SET
    products.embedding = (
      SELECT embeddings.values
      FROM SAFE.ML.PREDICT(
        MODEL EmbeddingsModel,
        (SELECT CONCAT('Name: ', name, ' \\nCategory: ', category, ' \\nBrand: ', brand, ' \\nDepartment: ', department) AS content)
      ) @{remote_udf_max_rows_per_rpc=200}
    ),
    products.embedding_model_version = 'text-embedding-005'
  WHERE products.id IN (SELECT p_sub.id
      FROM products AS p_sub
      WHERE p_sub.embedding IS NULL
      LIMIT 200)
  """

  run_spanner_query(sql)

  count_sql = "SELECT COUNT(*) AS to_embed FROM products WHERE products.embedding IS NULL"
  count_result = run_spanner_query(count_sql)
  rows_left = int(count_result.values[0][0])
  print(f"{rows_left} more rows left to embed...")

print("Embedding process is complete")

### Test Embedding Query

In [ ]:
search_phrase = "Luxury items for men"

sql = f"""WITH e AS (
    SELECT embeddings.values
    FROM ML.PREDICT(
    MODEL EmbeddingsModel, (
            SELECT '{search_phrase}' as content
        )
    )
)
SELECT COSINE_DISTANCE(
    products.embedding,
    e.values
  ) as dist,
  id,
  name,
  brand,
  department
FROM products, e
ORDER BY dist
LIMIT 5;
"""

run_spanner_query(sql)

## Setup Toolbox

### Deploy Toolbox to Cloud Run

Reference: https://github.com/googleapis/genai-toolbox/blob/main/docs/en/how-to/deploy_toolbox.md

#### Enable APIs

In [ ]:
! gcloud services enable run.googleapis.com \
                       cloudbuild.googleapis.com \
                       artifactregistry.googleapis.com \
                       iam.googleapis.com \
                       secretmanager.googleapis.com

#### Add Required Permissions for Cloud Build Account

In [ ]:
roles_array = [
    "roles/iam.serviceAccountCreator",
    "roles/secretmanager.admin",
    "roles/run.developer",
    "roles/iam.serviceAccountUser",
]

for r in roles_array:
  ! gcloud projects add-iam-policy-binding {project_id} \
      --member="serviceAccount:{project_number}-compute@developer.gserviceaccount.com" \
      --role="{r}"


#### Create a service account and add permissions

In [ ]:
! gcloud iam service-accounts create toolbox-identity

roles_array = [
    "roles/secretmanager.secretAccessor",
    "roles/spanner.viewer",
    "roles/spanner.databaseReader",
    "roles/spanner.databaseAdmin",
]

for r in roles_array:
  ! gcloud projects add-iam-policy-binding {project_id} \
      --member serviceAccount:toolbox-identity@{project_id}.iam.gserviceaccount.com \
      --role="{r}"

#### Configure tools.yaml

In [ ]:
# Reference: https://googleapis.github.io/genai-toolbox/resources/sources/spanner/
#            https://googleapis.github.io/genai-toolbox/resources/tools/
#            https://googleapis.github.io/genai-toolbox/resources/tools/spanner-sql/

import os
import json

tools_config = {
  "sources": {
    "spanner-ecom-source": {
        "kind": "spanner",
        "project": f"{project_id}",
        "instance": f"{spanner_instance_id}",
        "database": f"{spanner_database_id}",
        "dialect": "googlesql"
      }
    },
  "tools": {
    "get_products_by_natural_language": {
      "kind": "spanner-sql",
      "source": "spanner-ecom-source",
      "description": "Use this tool to look up product details from the product catalog. Availble data includes cost, category, brand, name, retail_price, department, and sku.",
      "statement": """WITH e AS (
    SELECT embeddings.values
    FROM ML.PREDICT(
    MODEL EmbeddingsModel, (
            SELECT @product_query as content
        )
    )
)
SELECT
  id,
  name,
  brand,
  department,
  cost,
  retail_price,
  sku
FROM products, e
ORDER BY COSINE_DISTANCE(
    products.embedding,
    e.values
  )
LIMIT 10;
""",
      "parameters": [
        {
          "name": "product_query",
          "type": "string",
          "description": "product_query is a short description of the product, including brand. Example: 'Seven7 Women's Long Sleeve Stripe Belted Top'"
        }
      ]
    }
  },
  "toolsets": {
    "default-toolset": [
      "get_products_by_natural_language"
    ]
  }
}

with open("tools.yaml", "w") as file:
    file.write(json.dumps(tools_config))


#### Create Secret Version from tools.yaml

In [ ]:
# Create the secret
! gcloud secrets create tools --data-file=tools.yaml

# Or update if it already exists
! gcloud secrets versions add tools --data-file=tools.yaml

#### Delete Local Copy of tools.yaml

In [ ]:
os.remove('tools.yaml')

#### Deploy Toolbox to Cloud Run

In [ ]:
# Define Toolbox Container Image
image = 'us-central1-docker.pkg.dev/database-toolbox/toolbox/toolbox:latest'

# Deploy to Cloud Run
! gcloud run deploy toolbox \
    --image {image} \
    --service-account toolbox-identity \
    --region {region} \
    --set-secrets "/app/tools.yaml=tools:latest" \
    --args="--tools_file=/app/tools.yaml","--address=0.0.0.0","--port=8080" \
    --allow-unauthenticated # https://cloud.google.com/run/docs/authenticating/public#gcloud


#### Test the Tool

##### Manually Invoke the Tool

In [ ]:
from toolbox_langchain import ToolboxClient

# Set the toolbox_url
toolbox_url = f"https://toolbox-{project_number}.{region}.run.app/"

# Replace with your Toolbox service's URL
toolbox = ToolboxClient("https://toolbox-797738608815.us-central1.run.app")

# Load all tools
tools = toolbox.load_toolset()

# Test the tool
result = await tools[0].arun({ "product_query": "Luxury items for men" })
result

##### Invoke the Tool via LangGraph

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_google_vertexai import ChatVertexAI
from langgraph.checkpoint.memory import MemorySaver
from toolbox_langchain import ToolboxClient

prompt = ''' You're a helpful AI assistant. You select the best Tool to get relevant data to answer the User's questions. '''

queries = [ "Show me products made by Seven7"]

# Load the tools from the Toolbox server
client = ToolboxClient(toolbox_url)
tools = client.load_toolset()

model = ChatVertexAI(model="gemini-2.0-flash")

agent = create_react_agent(model, tools, checkpointer=MemorySaver())

config = {"configurable": {"thread_id": "thread-1"}}
for query in queries:
    inputs = {"messages": [("user", prompt + query)]}
    response = agent.invoke(inputs, stream_mode="values", config=config)
    print(response["messages"][-1].content)

## Setup Agentspace Cloud Run Proxy

### Enable APIs

In [ ]:
! gcloud services enable run.googleapis.com cloudbuild.googleapis.com


### Create a Service Account

In [ ]:
# Reference: https://cloud.google.com/build/docs/deploying-builds/deploy-cloud-run#required_permissions
#            https://cloud.google.com/run/docs/deploying-source-code#required_roles

! gcloud iam service-accounts create agentspace-proxy

roles_array = [
    "roles/run.developer",
    "roles/logging.logWriter",
    "roles/artifactregistry.writer",
    "roles/iam.serviceAccountUser",
    "roles/storage.admin",
    "roles/aiplatform.user"
]

for r in roles_array:
  ! gcloud projects add-iam-policy-binding {project_id} \
      --member serviceAccount:agentspace-proxy@{project_id}.iam.gserviceaccount.com \
      --role="{r}"

### Add Required Permissions for Cloud Build Service Account

In [ ]:
! gcloud projects add-iam-policy-binding {project_id} \
      --member=serviceAccount:{project_number}-compute@developer.gserviceaccount.com \
      --role=roles/run.builder

### Define the Cloud Run Function

#### requirements.txt

In [ ]:
! mkdir -p cloud-run-source

requirements = """functions-framework==3.*
langgraph==0.3.21
langchain-google-vertexai==2.0.18
toolbox-langchain==0.1.0
"""

with open("cloud-run-source/requirements.txt", "w") as file:
    file.write(requirements)

#### main.py

In [ ]:
function_definition = """import functions_framework
from langgraph.prebuilt import create_react_agent
from langchain_google_vertexai import ChatVertexAI
from langgraph.checkpoint.memory import MemorySaver
from toolbox_langchain import ToolboxClient

@functions_framework.http
def invoke_toolbox(request):

    request_json = request.get_json(silent=True)
    request_args = request.args
    print("JSON:" + str(request_json))
    print("args:" + str(request_args))


    prompt = ''' You're a helpful AI assistant. You select the best Tool to get relevant data to answer the User's questions. '''

    product_query = request_json["product_query"]

    # Load the tools from the Toolbox server
    client = ToolboxClient('TOOLBOX_URL')
    tools = client.load_toolset()

    model = ChatVertexAI(model="gemini-2.0-flash")

    agent = create_react_agent(model, tools, checkpointer=MemorySaver())

    config = {"configurable": {"thread_id": "thread-1"}}

    inputs = {"messages": [("user", prompt + product_query)]}
    response = agent.invoke(inputs, stream_mode="values", config=config)

    response_message = response["messages"][-1].content

    return {"message": response_message}
"""

function_definition = function_definition.replace("TOOLBOX_URL", toolbox_url)

with open("cloud-run-source/main.py", "w") as file:
    file.write(function_definition)

### Deploy the Cloud Run Function

In [ ]:
! gcloud functions deploy agentspace-toolbox-proxy \
  --gen2 \
  --region={region} \
  --runtime=python312 \
  --source="./cloud-run-source" \
  --entry-point="invoke_toolbox" \
  --run-service-account="agentspace-proxy@{project_id}.iam.gserviceaccount.com" \
  --service-account="agentspace-proxy@{project_id}.iam.gserviceaccount.com" \
  --trigger-http \
  --allow-unauthenticated \
  --memory=2gi

### Test the Proxy

In [ ]:
agentspace_proxy_url = f"https://agentspace-toolbox-proxy-{project_number}.{region}.run.app"
request_body = {"product_query": "What products do we have in stock made by Seven7?"}

response = rest_api_helper(authed_session, agentspace_proxy_url, 'POST', request_body, {})
response

## Setup Agentspace

### Enable the `discoveryengine` API

In [ ]:
! gcloud services enable discoveryengine.googleapis.com

### Accept Terms

In [ ]:
# https://cloud.google.com/generative-ai-app-builder/docs/reference/rest/v1/projects/provision

url = f"https://discoveryengine.googleapis.com/v1/projects/{project_id}:provision"
request_body = {
  "acceptDataUseTerms": "true",
  "dataUseTermsVersion": "2022-11-23"
}
parameters = {}
result = rest_api_helper(authed_session, url, 'POST', request_body, parameters)
result

### Create Spanner Data Store

In [ ]:
! pip install google.cloud==0.34.0 google-cloud-discoveryengine==0.13.8

In [ ]:
# Reference: https://cloud.google.com/generative-ai-app-builder/docs/samples/genappbuilder-create-data-store

from google.api_core.client_options import ClientOptions
from google.cloud import discoveryengine

location = "global"
data_store_id = "spanner_chat_datastore"


def create_data_store(
    project_id: str,
    location: str,
    data_store_id: str,
) -> str:
    #  For more information, refer to:
    # https://cloud.google.com/generative-ai-app-builder/docs/locations#specify_a_multi-region_for_your_data_store
    client_options = (
        ClientOptions(api_endpoint=f"{location}-discoveryengine.googleapis.com")
        if location != "global"
        else None
    )

    # Create a client
    client = discoveryengine.DataStoreServiceClient(client_options=client_options)

    # The full resource name of the collection
    # e.g. projects/{project}/locations/{location}/collections/default_collection
    parent = client.collection_path(
        project=project_id,
        location=location,
        collection="default_collection",
    )

    data_store = discoveryengine.DataStore(
        display_name="Spanner Data Store",
        # Options: GENERIC, MEDIA, HEALTHCARE_FHIR
        industry_vertical=discoveryengine.IndustryVertical.GENERIC,
        # Options: SOLUTION_TYPE_RECOMMENDATION, SOLUTION_TYPE_SEARCH, SOLUTION_TYPE_CHAT, SOLUTION_TYPE_GENERATIVE_CHAT
        solution_types=[discoveryengine.SolutionType.SOLUTION_TYPE_CHAT],
        # TODO(developer): Update content_config based on data store type.
        # Options: NO_CONTENT, CONTENT_REQUIRED, PUBLIC_WEBSITE
        content_config=discoveryengine.DataStore.ContentConfig.CONTENT_REQUIRED,
    )

    request = discoveryengine.CreateDataStoreRequest(
        parent=parent,
        data_store_id=data_store_id,
        data_store=data_store,
        # Optional: For Advanced Site Search Only
        # create_advanced_site_search=True,
    )

    # Make the request
    operation = client.create_data_store(request=request)

    print(f"Waiting for operation to complete: {operation.operation.name}")
    response = operation.result()

    # After the operation is complete,
    # get information from operation metadata
    metadata = discoveryengine.CreateDataStoreMetadata(operation.metadata)

    # Handle the response
    print(response)
    print(metadata)

    return operation.operation.name

result = create_data_store(project_id, location, data_store_id)
result


### Import Data Store Documents from Spanner

In [ ]:
# Reference: https://cloud.google.com/generative-ai-app-builder/docs/samples/genappbuilder-import-documents-spanner

from google.api_core.client_options import ClientOptions
from google.cloud import discoveryengine

location = "global"
spanner_project_id = project_id

#  For more information, refer to:
# https://cloud.google.com/generative-ai-app-builder/docs/locations#specify_a_multi-region_for_your_data_store
client_options = (
    ClientOptions(api_endpoint=f"{location}-discoveryengine.googleapis.com")
    if location != "global"
    else None
)

# Create a client
client = discoveryengine.DocumentServiceClient(client_options=client_options)

# The full resource name of the search engine branch.
# e.g. projects/{project}/locations/{location}/dataStores/{data_store_id}/branches/{branch}
parent = client.branch_path(
    project=project_id,
    location=location,
    data_store=data_store_id,
    branch="default_branch",
)

request = discoveryengine.ImportDocumentsRequest(
    parent=parent,
    spanner_source=discoveryengine.SpannerSource(
        project_id=spanner_project_id,
        instance_id=spanner_instance_id,
        database_id=spanner_database_id,
        table_id=spanner_table_id,
    ),
    # Options: `FULL`, `INCREMENTAL`
    reconciliation_mode=discoveryengine.ImportDocumentsRequest.ReconciliationMode.FULL,
)

# Make the request
operation = client.import_documents(request=request)

#print(f"Waiting for operation to complete: {operation.operation.name}")
#response = operation.result()

# After the operation is complete,
# get information from operation metadata
#metadata = discoveryengine.ImportDocumentsMetadata(operation.metadata)

# Handle the response
print("Kicked off import of Spanner data.")


### Create an Agentspace Conversational app

In [ ]:
# Reference: https://cloud.google.com/dialogflow/cx/docs/reference/rest/v3beta1/projects.locations.agents/create?apix_params=%7B%22parent%22%3A%22projects%2F%7Bproject_id%7D%2Flocations%2Fglobal%22%2C%22resource%22%3A%7B%7D%7D
#            https://cloud.google.com/dialogflow/cx/docs/reference/rest/v3beta1/projects.locations.agents#Agent

url = f"https://dialogflow.googleapis.com/v3beta1/projects/{project_id}/locations/global/agents"
request_body = {
    "display_name": "Toolbox Agent",
    "default_language_code": "en",
    "time_zone": "America/Chicago",
    "start_playbook": f"projects/{project_id}/locations/glocal/agents/*/playbooks/00000000-0000-0000-0000-000000000000"
}
parameters = {}

result = rest_api_helper(authed_session, url, 'POST', request_body, parameters)
agent_id = result['name']
result

### Create an Agentspace Tool

In [ ]:
# Reference: https://cloud.google.com/dialogflow/cx/docs/reference/rest/v3beta1/projects.locations.agents.tools/create
#            https://cloud.google.com/dialogflow/cx/docs/reference/rest/v3beta1/projects.locations.agents.tools#Tool

url = f"https://dialogflow.googleapis.com/v3beta1/{agent_id}/tools"
request_body = {
    "display_name": "Database Toolbox",
    "description": "A tool to securely interact with Google databases through the Toolbox API",
    "tool_type": "CUSTOMIZED_TOOL",
    "open_api_spec": {
        "text_schema": f"""openapi: 3.0.0
info:
  title: Product Search API
  version: 1.0.0
servers:
  - url: '{agentspace_proxy_url}'
paths:
  /:
    post:
      summary: Search the product catalog
      operationId: searchProductCatalog
      requestBody:
        description: Product description to search
        required: true
        content:
          application/json:
            schema:
              $ref: '#/components/schemas/ProductSearch'
      responses:
        '200':
          description: Success
components:
  schemas:
    ProductSearch:
      type: object
      required:
        - product_query
      properties:
        product_query:
          type: string
"""
    }
}
parameters = {}

response = rest_api_helper(authed_session, url, 'POST', request_body, parameters)
agentspace_tool_id = response['name']
response

### Add permissions for Agentspace Tool to invoke Cloud Run

In [ ]:
roles_array = [
    "roles/run.invoker",
]

for r in roles_array:
  ! gcloud projects add-iam-policy-binding {project_id} \
      --member="serviceAccount:service-{project_number}@gcp-sa-dialogflow.iam.gserviceaccount.com" \
      --role="{r}"

### Create a Conversational Agent Playbook

In [ ]:
# Update the default playbook
url = f"https://dialogflow.googleapis.com/v3beta1/{agent_id}/playbooks/00000000-0000-0000-0000-000000000000"
request_body = {
    "displayName": "Invoke Toolbox API",
    "goal": "Help users lookup product data.",
    "referenced_tools": f"{agentspace_tool_id}",
    "instruction": {
        "guidelines": "Don't make anything up. Only provide factual information grounded in data provided by Tools and Data Sources. Always be polite and professional.",
        "steps": [
            {
                "text": "Ask the user to provide a description of the product they are searching for",  # Parent step text
                "steps": [            # Substeps nested directly under the parent
                    {"text": "Use ${TOOL:Database Toolbox}"},
                    {"text": "Respond to the user with a detailed response based on the output of the Tool"},

                ]
            }
            # Add more top-level steps here if needed
        ]
    }

}
parameters = {}

response = rest_api_helper(authed_session, url, 'PATCH', request_body, parameters)
response

## Manual Steps

The remaining steps require manual configuration in the console due to lack of API coverage this new Agentspace functionality. Instructions are taken from [this Qwiklab](https://partner.cloudskillsboost.google/course_templates/1191/labs/525477).

### Create an Agentspace app

In this task, you'll create a new Agentspace app using the "Enterprise Search and Assistant" template and integrating Google as the Identity Provider, while linking to a data store.

1. Navigate to **Agent Builder** > **Apps** > **+ Create App**.

2. Find the **Enterprise search and assistant** card and click **CREATE** to create an Agentspace app.

3. For an **app name**, enter `Products Agent`

4. For a **company name**, enter `Cymbal Shops`

5. Keep the location set to **global**.

6. Under **select tier** choose **Search + Assistant**.

7. Click **Continue**.

8. For an **Identity provider**, click **SELECT** on the **Google Identity Provider** card.

9. On the **Data** pane, select the **Travel Requests** data store you created above.

10. Click **Create**.

### Integrate your conversational agent with your Agentspace app

In this task, you'll grant your Agentspace assistant the ability to send messages to your conversational agent and receive its responses.

1. Navigate to **Agent Builder > Apps** and select `Products Agent` App.

2. From the left-hand navigation, select **Configurations**.

3. Select the **Assistant** tab.

4. Under the **Agents** header, select **Add an Item**. A card will be displayed to connect a **New Agent**:

5. Select your browser tab displaying your **Conversational Agents** console.

6. From the **Agent** dropdown at the top of the console, select **View all agents**.

7. At the end of your `Database Toolbox` agent's row, select the **Options icon** (three vertical dots) and select **Copy name**.

8. Navigate back to your **Agent Builder** tab, and in the **New Agent** card, paste the copied value in the **Agent** field.

9. For an **Agent display** name, use `Database Toolbox`.

10. For Instructions enter:

  ```
  Use for booking travel by providing a user, travel purpose, departure city, destination city, and a date range.
  ```

11. Notice that these instructions instruct the Agentspace assistant to do the work of gathering the required information before passing the details to the conversational agent for a single turn of conversation.

12. Click **Done**.

13. Click **Save and Publish** at the bottom of the pane.

### Communicate with your conversational agent through the Agentspace assistant

In this task, your Agentspace assistant will be able to communicate with the conversational agent, which will then utilize its tool to record travel requests.

> Note: It can take up to 10 minutes for your Agentspace app to be created. You can try the steps below, but if they don't proceed as expected, try to give your app more time to be created.

1. Select your **Agentspace app**. Navigate to the **Integration** tab from the left-hand navigation menu.

2. Under **The link to your web app** header, click **Open**. As stated at the start of this task, if you see a 404 error, you may need to give your app more time to be created. You can reload the page every few minutes until the Agentspace web app appears.

3. In the primary search bar, enter:

  ```
  Find product details for luxury gift items for women.
  ```

4. Your chat will be saved as a **Conversation** under the **Recents** header on the left-hand menu of the Agentspace web app.

5. Your assistant should have responded to your request. If you are asked any additional questions, use the Ask a follow-up field to reply to the assistant.

6. Under the assistant's responses, there is an **Options menu** (three vertical dots). Expand it and select **Show diagnostic info**.

7. In the diagnostic info displayed, you can view the metadata of the response, which includes `"functionName": "Database_Toolbox"`. This confirms that the Agentspace assistant invoked your conversational agent as a function call and received a response from it, which it has passed back to you.

8. Please note, in this activity you used very minimal Playbook instructions and no conversation examples, which means that this agent will not be very robust. If you need to restart the conversation to try it again, click the **New Conversation** button in the upper left.

## References

* [GenAI Toolbox](https://github.com/googleapis/genai-toolbox)